# Analytic law vs learned policy, side by side

Run both controllers on the **same** field and look at what each one does.

Everything is paired: given a seed, both controllers see byte-identical fields,
start positions, headings, and formation radii, so any difference you see is the
controller and not the draw.

**Where the numbers stand** (1000 paired held-out fields, see README):

| controller | success | vs analytic |
|---|---|---|
| tuned analytic | 66.5% | reference |
| PPO from scratch | 64.7% | -1.8%, CI [-4.7, +1.1] |
| PPO BC + fine-tune | 67.8% | +1.3%, CI [-1.5, +4.0] |
| either succeeds (oracle) | **76.5%** | **+10.0%, CI [+8.2, +11.9]** |

They are statistically tied overall but **complementary**: the policy wins
where the analytic law fails (`log_sum_exp`, `gaussian_pair`) and loses where
it is already near-perfect (`quadratic`, `rational_envelope`). The interesting
fields to try by hand are the ones where they disagree, and the last cell finds
those for you.

Unlike the other notebooks in this folder this one is **not** self-contained; it
imports the package next to it because it has to load trained checkpoints.

In [ ]:
import os, sys
import numpy as np
import matplotlib.pyplot as plt

# run from anywhere; point at the package directory
PKG = os.path.abspath("." if os.path.exists("compare_helpers.py")
                      else "rl_saddle_4robot")
os.chdir(PKG); sys.path.insert(0, PKG)

import compare_helpers as C
import saddle_fields as sf

print("field families:", ", ".join(sf.FAMILY_NAMES))
print()
print("available checkpoints (stem, use in POLICY below):")
for stem, _, _ in C.list_policies():
    if stem.endswith(("_best", "_final", "_bconly")) or "ckpt_15" in stem:
        print("   ", stem)

## Load the two controllers

`ppo_e4_ckpt_1511424` is the strongest checkpoint measured (67.8% on 1000
fields). `ppo_e2_best` is the from-scratch policy, which never saw the analytic
law at all, and is the more interesting one if you care whether PPO can find
this independently.

In [ ]:
# -------- EDIT THESE --------
POLICY = "ppo_e4_ckpt_1511424"   # any stem printed above
# ----------------------------

analytic = C.make_analytic()          # tuned rotating-Hessian, best swept gains
policy   = C.make_policy(POLICY)
print(analytic.label)
print(policy.label)

## Run one field

`FAMILY = None` draws from all eight. Set it to a name to force one, which is
how you go looking for a specific behaviour. `SEED` picks the field; change it
to get a different one.

In [ ]:
# -------- EDIT THESE --------
FAMILY = None        # None, or 'log_sum_exp', 'gaussian_pair', 'quadratic', ...
SEED   = 500000      # any integer; 500000+ are the held-out evaluation fields
# ----------------------------

res = C.run_pair(policy, analytic, SEED, FAMILY)
fld = res["field"]

print(f"field   : {fld.family}   saddle = ({fld.saddle[0]:+.3f}, {fld.saddle[1]:+.3f})")
print(f"          eigenvalues ({np.sort(fld.eigvals)[0]:+.2f}, {np.sort(fld.eigvals)[1]:+.2f})")
print()
for k in ("analytic", "policy"):
    r = res[k]["row"]
    mark = "OK " if r["success"] else "   "
    print(f"  {mark}{res[k]['label']:44s} final {r['e_final']:7.3f} m   "
          f"in-tol {r['time_in_tol']:5.1%}")

## Look at it

In [ ]:
def show(res, n_form=8):
    """Both trajectories over the field, plus distance against time."""
    fld = res["field"]
    fig, ax = plt.subplots(1, 2, figsize=(14.5, 6.2),
                           gridspec_kw={"width_ratios": [1.15, 1]})

    # -- field --------------------------------------------------------
    (x0, x1), (y0, y1) = fld.domain_bounds()
    xs = np.linspace(x0, x1, 140); ys = np.linspace(y0, y1, 140)
    X, Y = np.meshgrid(xs, ys)
    Z = np.vectorize(fld.phi)(X, Y)
    z0 = fld.phi(*fld.saddle)
    m = max(abs(np.nanmin(Z) - z0), abs(np.nanmax(Z) - z0), 1e-9)
    ax[0].contourf(X, Y, Z, levels=28, cmap="RdBu_r",
                   vmin=z0 - m, vmax=z0 + m, alpha=0.85)
    ax[0].contour(X, Y, Z, levels=28, colors="#8a8880", linewidths=0.35, alpha=0.5)

    for key, col in (("analytic", "#eb6834"), ("policy", "#2a78d6")):
        t = res[key]["log"]["centroid"]
        ax[0].plot(t[:, 0], t[:, 1], color=col, lw=2.2, zorder=9,
                   label=res[key]["label"])
        rb = res[key]["log"]["robots"]
        for i in range(0, len(rb), max(1, len(rb) // n_form)):
            q = rb[i][[0, 1, 2, 3, 0]]
            ax[0].plot(q[:, 0], q[:, 1], color=col, lw=0.8, alpha=0.45, zorder=8)
        ax[0].plot(*t[-1], marker="*", ms=16, color=col,
                   markeredgecolor="white", markeredgewidth=1.3, zorder=11)
    t0 = res["analytic"]["log"]["centroid"][0]
    ax[0].plot(*t0, marker="o", ms=9, color="#0b0b0b",
               markeredgecolor="white", markeredgewidth=1.3, zorder=12)
    ax[0].plot(*fld.saddle, marker="x", color="#0b0b0b", ms=14, mew=2.6, zorder=13)
    # nominal domain, so a trajectory that escaped it is unmistakable
    ax[0].add_patch(plt.Rectangle((x0, y0), x1 - x0, y1 - y0, fill=False,
                                  ec="#52514e", ls="--", lw=1.0, alpha=0.8,
                                  zorder=7))
    ax[0].set_aspect("equal"); ax[0].set_xticks([]); ax[0].set_yticks([])
    ax[0].set_title(f"{fld.family}   (circle = start, x = true saddle, "
                    f"star = end)", loc="left", fontsize=10)
    ax[0].legend(loc="upper right", fontsize=8.5, framealpha=0.9)

    # -- distance -----------------------------------------------------
    for key, col in (("analytic", "#eb6834"), ("policy", "#2a78d6")):
        e = res[key]["log"]["e"]
        ax[1].plot(np.arange(len(e)) * 0.1, e, color=col, lw=2.2,
                   label=res[key]["label"])
    ax[1].axhline(0.15, color="#0b0b0b", ls=":", lw=1.3)
    ax[1].text(1, 0.17, "success tolerance 0.15 m", fontsize=8, color="#52514e")
    ax[1].set_yscale("log"); ax[1].set_xlabel("time (s)")
    ax[1].set_ylabel("distance to true saddle (m)")
    ax[1].grid(alpha=0.25); ax[1].legend(fontsize=8.5)
    ax[1].set_title("Lower is better; flat means stuck", loc="left", fontsize=10)
    plt.tight_layout(); plt.show()

show(res)

## Find the fields where they disagree

This is the useful one. It runs a batch, then lists the fields where exactly one
controller succeeded, so you can drop those seeds into `SEED` above and watch
what actually differs.

In [ ]:
# -------- EDIT THESE --------
N_BATCH     = 40        # fields to scan
BATCH_SEED0 = 500000
BATCH_FAMILY = None     # or a single family name
# ----------------------------

rows = C.batch(policy, analytic, n=N_BATCH, seed0=BATCH_SEED0,
               family=BATCH_FAMILY)
C.batch_table(rows)

print()
print("  fields where ONLY THE POLICY succeeded (paste into SEED):")
for r in rows:
    if r["p_succ"] and not r["a_succ"]:
        print(f"    SEED = {r['seed']}   {r['family']:22s} "
              f"analytic {r['a_e']:6.2f} -> policy {r['p_e']:5.3f}")
print()
print("  fields where ONLY THE ANALYTIC LAW succeeded:")
for r in rows:
    if r["a_succ"] and not r["p_succ"]:
        print(f"    SEED = {r['seed']}   {r['family']:22s} "
              f"policy {r['p_e']:6.2f} -> analytic {r['a_e']:5.3f}")

## Notes on what you are looking at

- **Both stall on `gaussian_pair` and `streamfunction_quad`.** That is not a
  tuning failure. Those families put strong extrema inside the start annulus,
  and at any critical point the gradient is zero, so a 4-robot snapshot cannot
  tell "arrived at the saddle" from "stuck on a hilltop". The estimator's
  Hessian is always traceless (see `estimator.py`), so it always *looks* like a
  saddle. Fixing this needs a fifth robot at the centroid, or memory across
  time, not better gains.

- **The formation shrinks as it converges.** Both controllers learn that a
  smaller ring buys rotation authority: the feasible rotation band is
  `0.025/(0.3 R)` to `0.3/R` rad/s, so both bounds scale as `1/R`. See
  `outputs/figures/fig1_estimator_mechanism.png`.

- **Flat lines on the right panel mean stuck**, and the field panel will
  usually show the cluster parked on a coloured blob rather than the x.